Import the Libraries and set the environment variables

In [1]:
import os
import PyPDF2
from sentence_transformers import SentenceTransformer
import litellm
from litellm import completion
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from chromadb.config import Settings



/home/gaurav/AI_Practice_RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# # Set environment variables. Uncomment this if you want to set them directly.
#os.environ["HUGGINGFACE_TOKEN"] = "your_huggingface_token_here"
#os.environ["GEMINI_API_KEY"] = "your_gemini_api_key_here"
os.environ['LITELLM_LOG'] = 'DEBUG'

# print(os.environ)
# Retrieve environment variables from ~/.bashrc
HUGGINGFACE_TOKEN = os.getenv("HUGGINGFACE_TOKEN")
# print(HUGGINGFACE_TOKEN)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
# print(GEMINI_API_KEY)

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
# print(PINECONE_API_KEY)

In [3]:
print(os.getcwd())

/home/gaurav/AI_Practice_RAG/experiment_notebooks


Extract Text from folder containing PDF files

In [4]:
def extract_text_from_pdfs(folder_path):
    all_text = ""
    for filename in os.listdir(folder_path):
        if filename.endswith(".pdf"):
            file_path = os.path.join(folder_path, filename)
            with open(file_path, 'rb') as file:
                reader = PyPDF2.PdfReader(file)
                for page in reader.pages:
                    all_text += page.extract_text()
    return all_text

pdf_folder = "../dataset/"
all_text = extract_text_from_pdfs(pdf_folder)

Text Splitter

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # Size of each chunk
    chunk_overlap=50,  # Overlap between chunks to maintain context
    separators=["\n\n", "\n", " ", ""]  # Splitting hierarchy
)

chunks = text_splitter.split_text(all_text)

Set up the Knowledge Base with chromaDB and Generate Embeddings with sentence-transformers

In [6]:
# Initialize a persistent ChromaDB client
client = chromadb.PersistentClient(path="../chroma_db")

# Load the SentenceTransformer model for text embeddings
text_embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Delete existing collection (if needed)
try:
    client.delete_collection(name="knowledge_base")
    print("Deleted existing collection: knowledge_base")
except Exception as e:
    print(f"Collection does not exist or could not be deleted: {e}")

# Create a new collection for text embeddings
collection = client.create_collection(name="knowledge_base")

# Add text chunks to the collection
for i, chunk in enumerate(chunks):
    # Generate embeddings for the chunk
    embedding = text_embedding_model.encode(chunk)

    # Add to the collection with metadata
    collection.add(
        ids=[f"chunk_{i}"],  # Unique ID for each chunk
        embeddings=[embedding.tolist()],  # Embedding vector
        metadatas=[{"source": "pdf", "chunk_id": i}],  # Metadata
        documents=[chunk]  # Original text
    )

Deleted existing collection: knowledge_base


Perform Semantic Search with chromDB and Embedding Model

In [8]:
def semantic_search(query, top_k=5):
    # Generate embedding for the query
    query_embedding = text_embedding_model.encode(query)

    # Query the collection
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )
    return results

# Example query
query = "what is the role of methylation in Schizophrenia?"
results = semantic_search(query)

# Display results
for i, result in enumerate(results['documents'][0]):
    print(f"Result {i+1}: {result}\n")

Result 1: 5the causal factors may affect schizophrenia
through processes that have nothing to do with methylation(eg,possiblecausalmechanismsincludedisruptionofthelami-narorganizationofthecerebralcortex
72).Our RELNfindingpos-
sibly demonstrates the second possible model, in which themethylationstatusofdisease-relevantsitesinthebrainismir-roredbythecorrespondingsitesintheblood.Thus,previousstudies
19,22haveimplicatedmethylationsitesin RELNinschizo-

Result 2: Although the scope and quality of our phenotype data werelimited, smoking or other covariates did not account for thehypoxia findings. Although we can only speculate about thecause, we note that a substantial amount of literature existsshowingthathypoxiaduringfetaldevelopmentincreasestherisk of schizophrenia.
3It is known that environmentally in-
duced methylation changes can be preserved over a pro-longed period of time.
65,66One intriguing hypothesis is that

Result 3: 19,22Furthermore, support for a strong inverse corre-
lation

Generate Repsonse Based on Semantic Search

In [9]:
# Set up LiteLLM with Gemini

def generate_response(query, context):
    # Combine the query and context for the prompt
    prompt = f"Query: {query}\nContext: {context}\nAnswer:"

    # Call the Gemini model via LiteLLM
    response = completion(
        model="gemini/gemini-2.5-flash-lite",  # Use the Gemini model
        messages=[{"content": prompt, "role": "user"}],
        api_key= GEMINI_API_KEY
    )

    # Extract and return the generated text
    return response['choices'][0]['message']['content']

# Retrieve the top results from semantic search
search_results = semantic_search(query)
context = "\n".join(search_results['documents'][0])

# Generate a response using the retrieved context
response = generate_response(query, context)
print("Generated Response:\n", response)


Generated Response:
 Methylation plays a role in schizophrenia through at least two potential models:

*   **Indirect Causal Role:** Methylation in the blood may act as a "signature" that implicates a cause of the disease, even if that cause doesn't directly involve methylation in the brain. In this scenario, methylation in the blood is a marker, not the direct mechanism of disease progression in the brain.

*   **Direct Causal Role ("Functional Mirror Site"):** Methylation sites in the brain may have a direct causal role in schizophrenia. In this model, the methylation status of specific sites in the brain is mirrored by the corresponding sites in the blood. This means that changes in methylation in the brain lead to observed associations between schizophrenia and methylation patterns in blood.

The provided text specifically mentions that increased levels of methylation have been observed in schizophrenia cases in relation to the *RELN* gene. Previous studies have implicated methylat

Set up the Test Dataset for RAG Evaluation

In [10]:
import pandas as pd
#find it in the generated dataset folder if you miss it
df = pd.read_csv('../Evaluation/evaluation_results_question_methylation.csv')
print(df.shape)
print(df.head())
print(df.columns.to_list())

EmptyDataError: No columns to parse from file

Set the Prompt for Context Relevance Evaluation